# Exploratory Data Analysis: Stylized Facts of Financial Returns

This notebook presents empirical evidence for why simple Gaussian models (GBM) are inadequate for financial risk management, motivating the use of stochastic volatility models (Heston, Rough Heston) and copula-based dependence structures.

**Dataset:** 6-asset portfolio (IBM, AAPL, JPM, TLT, GLD, SPY), daily log-returns, Nov 2004 – Mar 2026 (~5,362 observations).

All figures and tables are generated by `src/python/eda/run_eda.py` and saved to `outputs/`.

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore', category=FutureWarning)

import pandas as pd
import numpy as np
from IPython.display import Image, display
from pathlib import Path

from python.utils.config import DATA_PROCESSED, REGIMES

FIG = Path('../outputs/figures/eda')
TBL = Path('../outputs/tables')

returns = pd.read_csv(DATA_PROCESSED / 'portfolio_returns.csv', index_col='Date', parse_dates=True)
print(f'Portfolio: {returns.shape[0]} days, {returns.shape[1]} assets, '
      f'{returns.index.min().date()} to {returns.index.max().date()}')

---
## Part 1: Univariate Return Statistics

### 1A. Summary Statistics

The table below compares each asset's distributional properties against the normal distribution baseline. Three key departures emerge:

1. **Negative skewness** — most equity returns are left-skewed (large losses more common than large gains).
2. **Excess kurtosis >> 0** — all assets exhibit heavy tails. Under normality, excess kurtosis = 0. We observe values of 3–18, indicating extreme events are far more probable than a Gaussian model predicts.
3. **Jarque-Bera p ≈ 0** — normality is overwhelmingly rejected for every asset.

In [ ]:
summary = pd.read_csv(TBL / 'summary_statistics.csv')
display(summary.style.format(precision=4).set_caption('Summary Statistics — Portfolio Assets'))

### 1B. Return Distribution Histograms

Each histogram overlays the empirical density with a fitted normal (black) and Student-t (red dashed). The Student-t distribution — with degrees of freedom typically 2–6 — captures the heavy tails far better. The annotations show that **1.0–1.9% of observations fall beyond ±3σ**, compared to the 0.27% expected under normality — a 4–7x excess.

In [ ]:
display(Image(filename=str(FIG / 'return_distributions_grid.png'), width=900))

### 1C. QQ-Plots

Quantile-quantile plots provide the clearest visual evidence of fat tails. If returns were normally distributed, all points would lie on the 45° line. Instead, we observe the characteristic **S-shaped deviation**: the left tail falls below the line (more extreme losses than predicted) and the right tail rises above it (more extreme gains). Points beyond ±2 theoretical quantiles are highlighted in red.

In [ ]:
display(Image(filename=str(FIG / 'qq_plots_grid.png'), width=900))

### 1D. Extreme Events Analysis

The most striking evidence: under normality, a 5σ event should occur once in ~3.5 million days (~14,000 years). In our 21-year sample, we observe **8–32 such events per asset** — ratios of 2,600x to 10,400x above the normal prediction. This alone disqualifies any Gaussian-based risk model for tail risk estimation.

In [ ]:
extreme = pd.read_csv(TBL / 'extreme_events.csv')
display(extreme.style.set_caption('Extreme Events: Actual vs Normal-Expected'))
print()
display(Image(filename=str(FIG / 'extreme_events_comparison.png'), width=900))

---
## Part 2: Volatility Dynamics

### 2A. Rolling Volatility

Financial volatility is **not constant** — it clusters in time, with long quiet periods punctuated by sharp spikes during crises. The plot below shows SPY's 21-day and 63-day rolling annualised volatility with crisis regimes shaded. The VIX (implied volatility index) on the right axis confirms the pattern. During the GFC (2008), annualised vol exceeded 80%; during COVID (2020), it spiked above 70%. In calm periods, it hovers around 10–15%.

This time-varying volatility is exactly what stochastic volatility models (Heston, Rough Heston) capture — and what GBM's constant σ cannot.

In [ ]:
display(Image(filename=str(FIG / 'rolling_volatility_spy.png'), width=900))

In [ ]:
display(Image(filename=str(FIG / 'rolling_volatility_all.png'), width=900))

### 2B. Volatility Clustering — Autocorrelation Evidence

This is the definitive test. We plot autocorrelation functions (ACF) for three transformations of SPY returns:

- **Raw returns** $r_t$: near-zero autocorrelation at all lags — returns are (approximately) unpredictable.
- **Absolute returns** $|r_t|$: strong, positive, *slowly decaying* autocorrelation — large moves cluster together.
- **Squared returns** $r_t^2$: similar pattern to $|r_t|$ — confirming ARCH/GARCH-type conditional heteroskedasticity.

**Interpretation:** You cannot predict the *direction* of tomorrow's return, but you *can* predict its *magnitude* — today's large move makes tomorrow's large move more likely. This is the volatility clustering phenomenon.

In [ ]:
display(Image(filename=str(FIG / 'acf_comparison_spy.png'), width=900))

### 2C. Ljung-Box Tests

The Ljung-Box test formalises the ACF visual. For **squared returns**, the test rejects the null of no autocorrelation at all lags with p-values indistinguishable from zero — confirming conditional heteroskedasticity across all assets. Gold (GLD) shows the weakest raw-return autocorrelation, consistent with efficient commodity markets.

In [ ]:
lb = pd.read_csv(TBL / 'ljung_box_tests.csv')
display(lb.style.set_caption('Ljung-Box Test Results'))

### 2D. Leverage Effect

The leverage effect — negative returns increase future volatility more than positive returns of equal magnitude — is visible in the asymmetric LOWESS curve. This asymmetry motivates the negative correlation parameter $\rho < 0$ between price and variance Brownian motions in the Heston model.

In [ ]:
display(Image(filename=str(FIG / 'leverage_effect.png'), width=900))

---
## Part 3: Multi-Asset Dependence Structure

### 3A. Correlation Matrices

The Pearson correlation matrix shows the expected structure: equities are positively correlated (0.40–0.72), bonds (TLT) are negatively correlated with equities, and gold (GLD) is near-uncorrelated — the diversification story. The small Pearson-Spearman differences (max ~0.05) suggest roughly linear dependence in the centre of the distribution, but this masks critical non-linear tail behaviour.

In [ ]:
display(Image(filename=str(FIG / 'correlation_matrices.png'), width=900))

### 3B. Rolling Correlations

Correlations are **not stable**. The 63-day rolling correlation between IBM and SPY oscillates between 0.3 and 0.8+, spiking during crises (GFC, COVID). The equity-bond pair (IBM–TLT) shows correlation ranging from -0.6 to +0.2, with the strongest negative correlation (best diversification) during rate-driven regimes. This time-varying dependence structure cannot be captured by a constant correlation matrix.

In [ ]:
display(Image(filename=str(FIG / 'rolling_correlations.png'), width=900))

### 3C. Tail Dependence — Why Diversification Fails in Crises

This is the most important dependence result. We condition on SPY being in its lower 5th percentile (market crash days), upper 95th percentile (rally days), or normal regime (25th–75th percentile), and measure pair correlations.

**Key findings:**
- **Equity–equity pairs** (IBM–AAPL, IBM–JPM, AAPL–JPM): correlations in the lower tail (0.37–0.42) are **5–10x higher** than in normal regimes (0.04–0.08). Diversification between equities evaporates exactly when you need it most.
- **SPY–TLT**: remains negatively correlated in tails — bonds provide genuine tail hedging.
- **SPY–GLD**: near-zero in all regimes — gold is regime-independent.

This asymmetric tail dependence is what vine copulas model and what Gaussian copulas miss entirely.

In [ ]:
display(Image(filename=str(FIG / 'tail_dependence.png'), width=900))

### 3D. Bivariate Scatter Plots

The scatter plots below colour-code observations from crisis regimes (red) vs calm periods (blue). During crises, the equity–equity clouds become tighter and more elongated along the diagonal — the correlation increase is visually striking. The marginal histograms show the heavier tails during crisis periods.

In [ ]:
display(Image(filename=str(FIG / 'bivariate_scatter_grid.png'), width=900))

---
## Part 4: Extended Dataset — Extreme Events in Historical Context

Using the longer IBM+SPY dataset (1993–2026, ~8,300 observations), we identify the 20 largest single-day moves for SPY. Many of these are **impossible events under normality**: the 2008-10-13 rally (+13.6%, 11.5σ) and the 2020-03-16 crash (−11.6%, 9.9σ) have normal-distribution probabilities that are computationally indistinguishable from zero — less than $10^{-15}$.

A risk model that assigns near-zero probability to events that occur every few years is not just imprecise — it is dangerous.

In [ ]:
top20 = pd.read_csv(TBL / 'top_20_extreme_days.csv')
display(top20.style.set_caption('Top 20 Extreme Days — SPY (1993–2026)'))
print()
display(Image(filename=str(FIG / 'extreme_events_timeline.png'), width=900))

---
## Part 5: Summary Dashboard

The six panels below synthesise the evidence. Together, they establish the empirical case for our modelling choices:

| Stylized Fact | Evidence | Model Implication |
|---|---|---|
| Fat tails | Excess kurtosis 3–18; 5σ events 2,600–10,000x more common than normal | Reject GBM → need heavy-tailed distributions |
| Volatility clustering | Strong ACF in \|r\| and r²; Ljung-Box p ≈ 0 | Need time-varying volatility → Heston / Rough Heston |
| Leverage effect | Asymmetric return-vol relationship | Need ρ < 0 in stochastic vol model |
| Tail dependence | Equity correlations 5–10x higher in crashes | Reject Gaussian copula → need vine copulas with Clayton family |
| Non-stationary correlations | Rolling correlations swing ±0.4 | Need dynamic dependence modelling |

In [ ]:
display(Image(filename=str(FIG / 'summary_dashboard.png'), width=1000))

---
## Conclusions

Every standard stylized fact of financial returns is confirmed in our dataset with overwhelming statistical significance:

1. **Normality is rejected** for all assets (Jarque-Bera p = 0). Fat tails are not subtle — 5σ events occur thousands of times more frequently than predicted.
2. **Volatility clusters** in time, with persistence visible out to 100+ lags in |r| and r² autocorrelation. GBM's constant σ is empirically untenable.
3. **The leverage effect** creates asymmetric risk — negative returns amplify future volatility, motivating ρ < 0 in stochastic vol models.
4. **Diversification fails in crises** — equity correlations surge from ~0.05 (normal) to ~0.40 (crash) in the lower tail, precisely when portfolio protection matters most.

These facts collectively justify our modelling strategy:
- **GBM** as a baseline (fast, analytically tractable, but misspecified)
- **Heston** for stochastic volatility with leverage
- **Rough Heston** for the empirical roughness of volatility paths (H ≈ 0.1)
- **Vine copulas** with Clayton family for lower-tail dependence

The *model risk* we aim to quantify is the gap in VaR/ES estimates between these progressively more realistic specifications.